# EXP37 FINAL CLEAN — Token + Arch2 Quantum Fusion + Fidelity Attention

This notebook is a clean regenerated file. It must not contain legacy config class references. Run from a fresh runtime.

In [ ]:
NOTEBOOK_VERSION = 'EXP37_FINAL_CLEAN_20260615_0918'
print('Loaded', NOTEBOOK_VERSION)
print('Use Runtime -> Disconnect and delete runtime, then Runtime -> Run all')

# EXP37 — Token + Arch2 Quantum Fusion + Fidelity-Guided Attention for FPSD

This notebook is generated with a **fresh-runtime safety design**.

Default mode is:

```python
DRY_RUN = True
```

In dry-run mode the notebook trains all registered model variants for a tiny synthetic pass, writes CSV logs, and runs `NOTEBOOK_SELF_TEST`. This validates architecture, tensor flow, metrics, and logging without requiring Google Drive/FPSD.

For the real FPSD run, set:

```python
DRY_RUN = False
```

and verify the FPSD/YOLO paths in `Exp37Config`.


In [ ]:
# Environment-driven runtime switches; safe defaults for fresh clones.
import os

def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    return default if value is None else value.strip().lower() in {"1", "true", "yes", "on"}

DRY_RUN = env_bool("POLSAR_DRY_RUN", True)
RUN_EXPERIMENT = env_bool("POLSAR_RUN_EXPERIMENT", True)
# Install Python packages with the documented requirements files, not during notebook execution.
INSTALL_MISSING_PACKAGES = False
print("Mode:", "synthetic smoke test" if DRY_RUN else "real FPSD", "| RUN_EXPERIMENT:", RUN_EXPERIMENT)


In [ ]:
# =========================
# CELL 2 — Dependency preflight / optional installs
# =========================
import sys, subprocess, importlib.util, os

print("INSTALL_MISSING_PACKAGES:", INSTALL_MISSING_PACKAGES)

def ensure(import_name, pip_name=None, required=False):
    """Return True if import is available. In full mode, install missing required packages automatically."""
    if importlib.util.find_spec(import_name) is not None:
        return True
    pkg = pip_name or import_name
    if INSTALL_MISSING_PACKAGES:
        print(f"Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        ok = importlib.util.find_spec(import_name) is not None
        if not ok and required:
            raise ImportError(f"Installed {pkg}, but import {import_name} still failed. Restart runtime and rerun all cells.")
        return ok
    msg = f"Package not installed: {import_name}."
    if required and not DRY_RUN:
        raise ImportError(msg + " Full FPSD mode requires it. Set INSTALL_MISSING_PACKAGES=True or run the install cell.")
    print(msg + " Skipping because this is DRY_RUN or optional.")
    return False

# Required only for full-FPSD mode. Dry-run uses synthetic tensors and avoids heavy installs.
HAS_ULTRALYTICS = ensure("ultralytics", "ultralytics", required=(not DRY_RUN))
HAS_COCO = ensure("pycocotools", "pycocotools", required=(not DRY_RUN))
HAS_RASTERIO = ensure("rasterio", "rasterio", required=(not DRY_RUN))
ensure("yaml", "pyyaml", required=False)

print("Dependency flags:", {"ultralytics": HAS_ULTRALYTICS, "pycocotools": HAS_COCO, "rasterio": HAS_RASTERIO})


In [ ]:
# =========================
# CELL 3 — Imports
# =========================
import os, json, math, time, random, hashlib, warnings, gc
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Tuple, Callable, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)
from scipy.stats import wilcoxon, ttest_1samp

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

# Inputs are local paths supplied through environment variables.
# If using Colab, mount Drive explicitly outside this notebook and set POLSAR_FPSD_ROOT, etc.

try:
    import rasterio
    from rasterio.windows import Window
    HAS_RASTERIO = True
except Exception:
    HAS_RASTERIO = False

print("DRY_RUN:", DRY_RUN)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| rasterio:", HAS_RASTERIO)


In [ ]:
# Portable path configuration: no user-specific, Colab, or machine-root paths.
def project_root() -> Path:
    return Path(os.environ.get("POLSAR_REPO_ROOT", os.getcwd())).expanduser().resolve()

def default_output_root() -> str:
    return os.environ.get("POLSAR_OUTPUT_ROOT", str(project_root() / "outputs"))

@dataclass
class Exp37Config:
    # Run mode
    dry_run: bool = DRY_RUN
    run_experiment: bool = RUN_EXPERIMENT

    # FPSD roots and detector paths. Used only when dry_run=False.
    fpsd_root: str = os.environ.get("POLSAR_FPSD_ROOT", str(project_root() / "data" / "FPSD"))
    output_root: str = default_output_root()
    exp31_run_folder: str = os.environ.get("POLSAR_EXP31_RUN_FOLDER", "")
    yolo_best: str = os.environ.get("POLSAR_YOLO_BEST", "")
    yolo_data_yaml: str = os.environ.get("POLSAR_YOLO_DATA_YAML", "")

    # Auto-resolved split paths. Used only when dry_run=False.
    train_image_dir: str = os.environ.get("POLSAR_TRAIN_IMAGE_DIR", "")
    train_tif_dir: str = os.environ.get("POLSAR_TRAIN_TIF_DIR", "")
    train_ann: str = os.environ.get("POLSAR_TRAIN_ANN", "")
    val_image_dir: str = os.environ.get("POLSAR_VAL_IMAGE_DIR", "")
    val_tif_dir: str = os.environ.get("POLSAR_VAL_TIF_DIR", "")
    val_ann: str = os.environ.get("POLSAR_VAL_ANN", "")

    # Candidate extraction / PolSAR settings
    imgsz: int = 640
    yolo_conf: float = 0.001
    yolo_iou: float = 0.65
    max_det: int = 300
    iou_eval: float = 0.50
    chip_size: int = 64
    patch_grid: int = 6
    c9_layout: str = "fpsd_exp25_9"
    symmetrize_c9: bool = True
    eps: float = 1e-8
    token_dim: int = 8

    # Arch2 quantum Bloch encoding
    q_layers: int = 2
    arch2_obs_dim: int = 12

    # Shared heads
    d_model: int = 32
    heads: int = 4
    dropout: float = 0.10
    control_hidden_a2: int = 4
    fidelity_init_alpha: float = 0.10
    fidelity_center: bool = True

    # Training defaults. These are overridden below for dry-run.
    seeds: Tuple[int, ...] = tuple(range(42, 52))
    epochs: int = 30
    batch_size: int = 128
    lr: float = 2e-3
    weight_decay: float = 1e-4
    pos_weight: float = -1.0  # -1 => auto from train split

    # Synthetic dry-run size
    n_synth_train: int = 96
    n_synth_val: int = 64

CFG = Exp37Config()

# Dry-run must be fast enough for Runtime -> Run all.
if CFG.dry_run:
    CFG.seeds = (42,)
    CFG.epochs = 0
    CFG.batch_size = 8
    CFG.n_synth_train = 16
    CFG.n_synth_val = 12
    CFG.output_root = default_output_root()

RUN_ID = time.strftime("exp37_fusion_fidelity_%Y%m%d_%H%M%S")
RUN_DIR = Path(CFG.output_root) / RUN_ID
CACHE_DIR = Path(CFG.output_root) / "exp37_chip_cache"
for d in [RUN_DIR / "logs", RUN_DIR / "ckpt", CACHE_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

with open(RUN_DIR / "logs" / "config_initial.json", "w") as f:
    json.dump(asdict(CFG), f, indent=2)

print("RUN_DIR:", RUN_DIR)
print("DRY_RUN config: seeds=", CFG.seeds, "epochs=", CFG.epochs, "batch_size=", CFG.batch_size)


In [ ]:
# =========================
# CELL 5 — Path resolution for full FPSD mode
# =========================
def _exists(p: str) -> bool:
    return bool(p) and Path(p).exists()

def _first_existing(candidates):
    for c in candidates:
        if Path(c).exists():
            return str(c)
    return ""

def _latest_glob(root: str, pattern: str) -> str:
    hits = sorted(Path(root).glob(pattern), key=lambda x: x.stat().st_mtime, reverse=True)
    return str(hits[0]) if hits else ""

def resolve_paths(cfg: Exp37Config) -> Exp37Config:
    exp31 = Path(cfg.exp31_run_folder) if cfg.exp31_run_folder else None
    if not cfg.yolo_best:
        cfg.yolo_best = _first_existing([
            exp31 / "stage1_predictions" / "autotrain_yolov8n" / "weights" / "best.pt"
        ]) if exp31 else ""
    if not cfg.yolo_data_yaml:
        cfg.yolo_data_yaml = _first_existing([
            exp31 / "stage1_predictions" / "fpsd_stage1_data.yaml"
        ]) if exp31 else ""

    co = Path(cfg.fpsd_root) / "coco"
    if not cfg.train_image_dir:
        cfg.train_image_dir = _first_existing([co / "train2017_jpg", co / "train2017_tif"])
    if not cfg.train_tif_dir:
        cfg.train_tif_dir = _first_existing([co / "train2017_tif", co / "train"])
    if not cfg.train_ann:
        cfg.train_ann = _first_existing([co / "annotations" / "instances_train2017_tif.json", co / "annotations" / "instances_train2017.json"])
    if not cfg.val_image_dir:
        cfg.val_image_dir = _first_existing([co / "val2017_jpg", co / "val2017_tif"])
    if not cfg.val_tif_dir:
        cfg.val_tif_dir = _first_existing([co / "val2017_tif", co / "val"])
    if not cfg.val_ann:
        cfg.val_ann = _first_existing([co / "annotations" / "instances_val2017_tif.json", co / "annotations" / "instances_val2017.json"])
    return cfg

if CFG.dry_run:
    print("DRY_RUN=True: skipping FPSD/YOLO path assertions.")
    print("NOT TESTED: real Google Drive/FPSD paths and YOLO candidate generation.")
else:
    CFG = resolve_paths(CFG)
    for key in ["yolo_best", "yolo_data_yaml", "train_image_dir", "train_tif_dir", "train_ann", "val_image_dir", "val_tif_dir", "val_ann"]:
        value = getattr(CFG, key)
        print(f"{key:16s}: {value} | exists={_exists(value)}")
    required_paths = ["yolo_best", "train_image_dir", "train_tif_dir", "train_ann",
                      "val_image_dir", "val_tif_dir", "val_ann"]
    missing = [name for name in required_paths if not _exists(getattr(CFG, name))]
    if missing:
        raise FileNotFoundError(
            f"Full FPSD mode requires verified train/val inputs: {missing}. "
            "Set POLSAR_* environment variables. No validation-to-training fallback is permitted."
        )

with open(RUN_DIR / "logs" / "config_resolved.json", "w") as f:
    json.dump(asdict(CFG), f, indent=2)


In [ ]:
# =========================
# CELL 6 — PolSAR C9/rho/token extraction utilities
# =========================
def raw_to_c9(raw, cfg: Exp37Config):
    raw = np.asarray(raw)
    if raw.ndim == 2:
        raw = raw[:, :, None]
    if raw.shape[-1] < 9:
        raise ValueError(f"Expected at least 9 channels for C9, got shape {raw.shape}")

    layout = cfg.c9_layout.lower()
    C = np.zeros(raw.shape[:2] + (3, 3), dtype=np.complex64)

    if layout in ["fpsd_exp25_9", "c11_c12r_c12i_c13r_c13i_c22_c23r_c23i_c33"]:
        C11 = raw[:, :, 0].astype(np.float32)
        C12 = raw[:, :, 1].astype(np.float32) + 1j * raw[:, :, 2].astype(np.float32)
        C13 = raw[:, :, 3].astype(np.float32) + 1j * raw[:, :, 4].astype(np.float32)
        C22 = raw[:, :, 5].astype(np.float32)
        C23 = raw[:, :, 6].astype(np.float32) + 1j * raw[:, :, 7].astype(np.float32)
        C33 = raw[:, :, 8].astype(np.float32)
    elif layout in ["compact9", "compact_3diag_3offdiag_realimag"]:
        C11 = raw[:, :, 0].astype(np.float32)
        C22 = raw[:, :, 1].astype(np.float32)
        C33 = raw[:, :, 2].astype(np.float32)
        C12 = raw[:, :, 3].astype(np.float32) + 1j * raw[:, :, 4].astype(np.float32)
        C13 = raw[:, :, 5].astype(np.float32) + 1j * raw[:, :, 6].astype(np.float32)
        C23 = raw[:, :, 7].astype(np.float32) + 1j * raw[:, :, 8].astype(np.float32)
    else:
        raise ValueError(f"Unsupported c9_layout={cfg.c9_layout}")

    C[:, :, 0, 0] = C11
    C[:, :, 1, 1] = C22
    C[:, :, 2, 2] = C33
    C[:, :, 0, 1] = C12; C[:, :, 1, 0] = np.conj(C12)
    C[:, :, 0, 2] = C13; C[:, :, 2, 0] = np.conj(C13)
    C[:, :, 1, 2] = C23; C[:, :, 2, 1] = np.conj(C23)

    if cfg.symmetrize_c9:
        C = 0.5 * (C + np.swapaxes(np.conj(C), -1, -2))
        for d in range(3):
            C[:, :, d, d] = np.maximum(np.real(C[:, :, d, d]), 0.0).astype(np.float32)
    return C.astype(np.complex64)

def safe_corr(Cij, Cii, Cjj, eps=1e-8):
    return Cij / np.sqrt(np.maximum(Cii * Cjj, eps))

def patch_coherency_features(c9_patch, cfg: Exp37Config):
    C = np.nanmean(c9_patch.reshape(-1, 3, 3), axis=0).astype(np.complex64)
    C = 0.5 * (C + C.conj().T)
    diag = np.maximum(np.real(np.diag(C)), 0.0)
    span = float(np.sum(diag))
    eps = float(cfg.eps)

    if span <= eps or not np.isfinite(span):
        return np.zeros((8,), np.float32), (np.eye(3, dtype=np.complex64) / 3.0)

    rho = (C / (span + eps)).astype(np.complex64)
    C11, C22, C33 = diag
    C12, C13, C23 = C[0, 1], C[0, 2], C[1, 2]
    g12 = safe_corr(C12, C11, C22, eps)
    g13 = safe_corr(C13, C11, C33, eps)
    g23 = safe_corr(C23, C22, C33, eps)

    ev = np.clip(np.linalg.eigvalsh(rho).real, 0, None)
    ev = ev / (ev.sum() + eps)
    entropy = -np.sum(ev * np.log(ev + eps)) / np.log(3.0)

    token = np.array([
        np.log1p(span),
        np.clip(abs(g12), 0, 5), np.clip(abs(g13), 0, 5), np.clip(abs(g23), 0, 5),
        np.angle(g12) / np.pi, np.angle(g13) / np.pi, np.angle(g23) / np.pi,
        entropy,
    ], dtype=np.float32)
    return np.nan_to_num(token, nan=0.0, posinf=0.0, neginf=0.0), rho

def chip_to_tokens(c9_chip, cfg: Exp37Config):
    H, W = c9_chip.shape[:2]
    G = int(cfg.patch_grid)
    ph, pw = H // G, W // G
    tokens, rhos = [], []
    for gy in range(G):
        for gx in range(G):
            y1, y2 = gy * ph, (gy + 1) * ph if gy < G - 1 else H
            x1, x2 = gx * pw, (gx + 1) * pw if gx < G - 1 else W
            tok, rho = patch_coherency_features(c9_chip[y1:y2, x1:x2], cfg)
            tokens.append(tok); rhos.append(rho)
    return np.stack(tokens).astype(np.float32), np.stack(rhos).astype(np.complex64)

class C9ImageReader:
    def __init__(self, path, cfg: Exp37Config):
        self.path = Path(path)
        self.cfg = cfg
        self._src = None
        self._full = None
        if HAS_RASTERIO:
            self._src = rasterio.open(self.path)
            self.width = self._src.width
            self.height = self._src.height
        else:
            import tifffile
            arr = tifffile.imread(str(self.path))
            if arr.ndim == 2:
                arr = arr[:, :, None]
            elif arr.ndim == 3 and arr.shape[0] <= 32 and arr.shape[0] < arr.shape[-1]:
                arr = np.moveaxis(arr, 0, -1)
            self._full = arr
            self.height, self.width = arr.shape[:2]

    def close(self):
        if self._src is not None:
            self._src.close()

    def _read_raw(self, x1, y1, x2, y2):
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(self.width, int(x2)), min(self.height, int(y2))
        if self._src is not None:
            return np.moveaxis(self._src.read(window=Window(x1, y1, x2 - x1, y2 - y1)), 0, -1)
        return self._full[y1:y2, x1:x2, :]

    def read_c9_chip(self, cx, cy, chip_size):
        half = chip_size // 2
        x1, y1 = int(round(cx - half)), int(round(cy - half))
        x2, y2 = x1 + chip_size, y1 + chip_size
        raw = self._read_raw(x1, y1, x2, y2)
        c9 = raw_to_c9(raw, self.cfg)
        out = np.zeros((chip_size, chip_size, 3, 3), np.complex64)
        h, w = c9.shape[:2]
        out[:h, :w] = c9[:chip_size, :chip_size]
        return out

print("PolSAR token/rho utilities ready.")


In [ ]:
# =========================
# CELL 7 — Dataset builders: synthetic dry-run + full FPSD hooks
# =========================
def _random_density(n, T, rng):
    A = rng.standard_normal((n, T, 3, 3)) + 1j * rng.standard_normal((n, T, 3, 3))
    rho = A @ np.conj(np.swapaxes(A, -1, -2))
    tr = np.real(np.einsum("...ii->...", rho))[..., None, None]
    rho = rho / np.maximum(tr, 1e-8)
    return rho.astype(np.complex64)

def _tokens_from_rho_numpy(rho, eps=1e-8):
    # Synthetic counterpart of the handcrafted token: span proxy + pair magnitudes/phases + entropy.
    diag = np.maximum(np.real(np.diagonal(rho, axis1=-2, axis2=-1)), 0.0)
    C11, C22, C33 = diag[..., 0], diag[..., 1], diag[..., 2]
    C12, C13, C23 = rho[..., 0, 1], rho[..., 0, 2], rho[..., 1, 2]
    g12 = C12 / np.sqrt(np.maximum(C11 * C22, eps))
    g13 = C13 / np.sqrt(np.maximum(C11 * C33, eps))
    g23 = C23 / np.sqrt(np.maximum(C22 * C33, eps))
    ev = np.clip(np.linalg.eigvalsh(rho).real, 0, None)
    ev = ev / np.maximum(ev.sum(axis=-1, keepdims=True), eps)
    entropy = -np.sum(ev * np.log(ev + eps), axis=-1) / np.log(3.0)
    span_proxy = np.ones_like(entropy)
    tok = np.stack([
        np.log1p(span_proxy),
        np.clip(np.abs(g12), 0, 5), np.clip(np.abs(g13), 0, 5), np.clip(np.abs(g23), 0, 5),
        np.angle(g12) / np.pi, np.angle(g13) / np.pi, np.angle(g23) / np.pi,
        entropy,
    ], axis=-1).astype(np.float32)
    return np.nan_to_num(tok, nan=0.0, posinf=0.0, neginf=0.0)

def build_synthetic_dataset(cfg: Exp37Config):
    rng = np.random.default_rng(123)
    T = cfg.patch_grid ** 2
    out = {}
    for split, n in [("train", cfg.n_synth_train), ("val", cfg.n_synth_val)]:
        rho = _random_density(n, T, rng)
        tokens = _tokens_from_rho_numpy(rho, cfg.eps)
        # Label is deliberately learnable from both density purity and token coherence.
        ev = np.linalg.eigvalsh(rho).real
        purity = ev[..., -1].mean(axis=1)
        coherence = tokens[..., 1:4].mean(axis=(1, 2))
        score = 0.65 * purity + 0.35 * coherence + 0.03 * rng.standard_normal(n)
        y = (score > np.median(score)).astype(np.float32)
        out[split] = (rho.astype(np.complex64), tokens.astype(np.float32), y)
    return out

def load_coco_gt(annotation_json):
    with open(annotation_json, "r") as f:
        coco = json.load(f)
    stem_to_id, name_to_id = {}, {}
    for im in coco.get("images", []):
        name = im.get("file_name", "")
        name_to_id[name] = int(im["id"])
        stem_to_id[Path(name).stem] = int(im["id"])
    gt = {int(im["id"]): [] for im in coco.get("images", [])}
    for a in coco.get("annotations", []):
        if a.get("iscrowd", 0):
            continue
        x, y, w, h = a.get("bbox", [0, 0, 0, 0])[:4]
        if w > 0 and h > 0:
            gt.setdefault(int(a["image_id"]), []).append([float(x), float(y), float(x + w), float(y + h)])
    gt = {k: np.asarray(v, np.float32).reshape(-1, 4) for k, v in gt.items()}
    return stem_to_id, name_to_id, gt

def iou_one_to_many(box, boxes):
    boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
    if len(boxes) == 0:
        return np.zeros((0,), np.float32)
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    a1 = max(0, box[2] - box[0]) * max(0, box[3] - box[1])
    a2 = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
    return inter / np.maximum(a1 + a2 - inter, 1e-9)

def cache_key(cfg: Exp37Config, split: str):
    sig = f"EXP37|{split}|{cfg.chip_size}|{cfg.patch_grid}|{cfg.c9_layout}|{cfg.yolo_conf}|{cfg.yolo_iou}|{cfg.max_det}|{cfg.iou_eval}"
    return CACHE_DIR / f"chips_{split}_{hashlib.md5(sig.encode()).hexdigest()[:10]}.npz"

def yolo_candidates(yolo, image_dir, stem_to_id, name_to_id, gt, cfg: Exp37Config):
    rows = []
    result_iter = yolo.predict(
        source=image_dir, imgsz=cfg.imgsz, conf=cfg.yolo_conf, iou=cfg.yolo_iou,
        max_det=cfg.max_det, save=False, verbose=False, stream=True,
    )
    for res in result_iter:
        p = Path(res.path)
        image_id = stem_to_id.get(p.stem, name_to_id.get(p.name))
        if image_id is None or res.boxes is None or len(res.boxes) == 0:
            continue
        xyxy = res.boxes.xyxy.detach().cpu().numpy()
        for b in xyxy:
            x1, y1, x2, y2 = map(float, b)
            ious = iou_one_to_many([x1, y1, x2, y2], gt.get(int(image_id), np.zeros((0, 4), np.float32)))
            label = int((float(ious.max()) if len(ious) else 0.0) >= cfg.iou_eval)
            rows.append(dict(stem=p.stem, image_id=int(image_id), x1=x1, y1=y1, x2=x2, y2=y2, candidate_label=label))
    return pd.DataFrame(rows)

def extract_split_from_candidates(df, tif_dir, cfg: Exp37Config):
    R, Toks, Y = [], [], []
    for stem, group in df.groupby("stem"):
        tif_path = ""
        for ext in [".tif", ".tiff", ".TIF", ".TIFF"]:
            p = Path(tif_dir) / f"{stem}{ext}"
            if p.exists():
                tif_path = str(p); break
        if not tif_path:
            continue
        reader = C9ImageReader(tif_path, cfg)
        try:
            for _, row in group.iterrows():
                cx = (row.x1 + row.x2) / 2.0
                cy = (row.y1 + row.y2) / 2.0
                tok, rho = chip_to_tokens(reader.read_c9_chip(cx, cy, cfg.chip_size), cfg)
                R.append(rho); Toks.append(tok); Y.append(int(row.candidate_label))
        finally:
            reader.close()
    if not R:
        raise RuntimeError("No C9 chips extracted. Check TIF path/layout and YOLO candidates.")
    return np.stack(R).astype(np.complex64), np.stack(Toks).astype(np.float32), np.asarray(Y, np.float32)

def get_real_split(yolo, split: str, cfg: Exp37Config):
    ck = cache_key(cfg, split)
    if ck.exists():
        z = np.load(ck)
        print(f"[cache] loaded {split}: {ck}")
        return z["rho"], z["tokens"], z["y"]
    if split == "train":
        image_dir, tif_dir, ann = cfg.train_image_dir, cfg.train_tif_dir, cfg.train_ann
    elif split == "val":
        image_dir, tif_dir, ann = cfg.val_image_dir, cfg.val_tif_dir, cfg.val_ann
    else:
        raise ValueError(split)
    stem_to_id, name_to_id, gt = load_coco_gt(ann)
    df = yolo_candidates(yolo, image_dir, stem_to_id, name_to_id, gt, cfg)
    df.to_csv(RUN_DIR / "logs" / f"candidates_{split}.csv", index=False)
    rho, tokens, y = extract_split_from_candidates(df, tif_dir, cfg)
    np.savez_compressed(ck, rho=rho, tokens=tokens, y=y)
    print(f"[cache] wrote {split}: {ck}")
    return rho, tokens, y

def build_dataset(cfg: Exp37Config):
    if cfg.dry_run:
        print("Building synthetic DRY_RUN dataset.")
        return build_synthetic_dataset(cfg)

    if not HAS_ULTRALYTICS:
        raise ImportError(
            "Full FPSD mode requires ultralytics. The notebook now auto-installs it when DRY_RUN=False. "
            "Restart runtime, run from Cell 1, and confirm INSTALL_MISSING_PACKAGES=True."
        )
    from ultralytics import YOLO
    yolo = YOLO(cfg.yolo_best)
    data = {"val": get_real_split(yolo, "val", cfg)}
    if _exists(cfg.train_ann) and _exists(cfg.train_tif_dir) and _exists(cfg.train_image_dir):
        data["train"] = get_real_split(yolo, "train", cfg)
    else:
        raise FileNotFoundError(
            "FPSD training inputs missing. Refusing to split validation candidates into training data."
        )
    return data

DATA = build_dataset(CFG)
for split, (rho, tokens, y) in DATA.items():
    print(f"{split:5s}: rho={rho.shape}, tokens={tokens.shape}, y={y.shape}, pos_rate={float(y.mean()):.3f}")

if CFG.pos_weight < 0:
    ytr = DATA["train"][2]
    n_pos = float(ytr.sum())
    n_neg = float(len(ytr) - n_pos)
    CFG.pos_weight = n_neg / max(n_pos, 1.0)
    print("Auto pos_weight:", CFG.pos_weight)


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
import numpy as np
import pandas as pd
from pathlib import Path

def yolo_candidates_conf_only(yolo, image_dir, stem_to_id, name_to_id, gt, cfg):
    rows = []

    result_iter = yolo.predict(
        source=image_dir,
        imgsz=cfg.imgsz,
        conf=cfg.yolo_conf,
        iou=cfg.yolo_iou,
        max_det=cfg.max_det,
        save=False,
        verbose=False,
        stream=True,
    )

    for res in result_iter:
        p = Path(res.path)
        image_id = stem_to_id.get(p.stem, name_to_id.get(p.name))

        if image_id is None or res.boxes is None or len(res.boxes) == 0:
            continue

        xyxy = res.boxes.xyxy.detach().cpu().numpy()
        confs = res.boxes.conf.detach().cpu().numpy()

        for b, conf in zip(xyxy, confs):
            x1, y1, x2, y2 = map(float, b)

            ious = iou_one_to_many(
                [x1, y1, x2, y2],
                gt.get(int(image_id), np.zeros((0, 4), np.float32)),
            )

            label = int((float(ious.max()) if len(ious) else 0.0) >= cfg.iou_eval)

            rows.append({
                "stem": p.stem,
                "image_id": int(image_id),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "det_conf": float(conf),
                "candidate_label": label,
            })

    return pd.DataFrame(rows)


def candidate_score_metrics(y_true, score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    y_pred = (score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    fpr = fp / (fp + tn + 1e-12)
    fa_rej = tn / (tn + fp + 1e-12)

    return {
        "AP": average_precision_score(y_true, score),
        "AUC": roc_auc_score(y_true, score),
        "F1@0.5": f1_score(y_true, y_pred, zero_division=0),
        "Precision@0.5": precision_score(y_true, y_pred, zero_division=0),
        "Recall@0.5": recall_score(y_true, y_pred, zero_division=0),
        "FPR@0.5": fpr,
        "FA-Rej@0.5": fa_rej,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
    }


# Run detector-confidence-only baseline on validation split
if not CFG.dry_run:
    stem_to_id, name_to_id, gt = load_coco_gt(CFG.val_ann)
    yolo = YOLO(CFG.yolo_best)

    df_conf_val = yolo_candidates_conf_only(
        yolo=yolo,
        image_dir=CFG.val_image_dir,
        stem_to_id=stem_to_id,
        name_to_id=name_to_id,
        gt=gt,
        cfg=CFG,
    )

    out_csv = RUN_DIR / "logs" / "detector_confidence_val_candidates.csv"
    df_conf_val.to_csv(out_csv, index=False)

    print("Saved:", out_csv)
    print("Validation candidates:", len(df_conf_val))
    print("Validation positives:", int(df_conf_val["candidate_label"].sum()))
    print("Validation false alarms:", int(len(df_conf_val) - df_conf_val["candidate_label"].sum()))

    metrics_conf = candidate_score_metrics(
        df_conf_val["candidate_label"].values,
        df_conf_val["det_conf"].values,
        threshold=0.5,
    )

    metrics_conf_df = pd.DataFrame([metrics_conf])
    metrics_csv = RUN_DIR / "logs" / "detector_confidence_only_metrics.csv"
    metrics_conf_df.to_csv(metrics_csv, index=False)

    print("Saved:", metrics_csv)
    display(metrics_conf_df)
else:
    print("Detector confidence baseline: skipped for synthetic smoke test.")


In [ ]:
# =========================
# CELL 8 — Model zoo: Token baseline, Arch2 quantum, fusion, fidelity attention
# =========================
I2 = torch.tensor([[1, 0], [0, 1]], dtype=torch.complex64)
X2 = torch.tensor([[0, 1], [1, 0]], dtype=torch.complex64)
Y2 = torch.tensor([[0, -1j], [1j, 0]], dtype=torch.complex64)
Z2 = torch.tensor([[1, 0], [0, -1]], dtype=torch.complex64)
PAULI_1Q = {"I": I2, "X": X2, "Y": Y2, "Z": Z2}

def ry(angle):
    c = torch.cos(angle / 2)
    s = torch.sin(angle / 2)
    mat = torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)
    return torch.complex(mat, torch.zeros_like(mat))

def rz(angle):
    e0 = torch.complex(torch.cos(-angle / 2), torch.sin(-angle / 2))
    e1 = torch.complex(torch.cos(angle / 2), torch.sin(angle / 2))
    z = torch.zeros_like(e0)
    return torch.stack([torch.stack([e0, z], -1), torch.stack([z, e1], -1)], -2)

def pauli3(name: str):
    mat = PAULI_1Q[name[0]]
    for ch in name[1:]:
        mat = torch.kron(mat, PAULI_1Q[ch])
    return mat

ARCH2_OBS_NAMES = ["XII", "YII", "ZII", "IXI", "IYI", "IZI", "IIX", "IIY", "IIZ", "ZZI", "ZIZ", "IZZ"]
ARCH2_OBS = torch.stack([pauli3(s) for s in ARCH2_OBS_NAMES], dim=0)
PAIRS = [(0, 1), (0, 2), (1, 2)]

def kron3(a, b, c):
    ab = torch.einsum("...ij,...kl->...ikjl", a, b).reshape(*a.shape[:-2], 4, 4)
    return torch.einsum("...ij,...kl->...ikjl", ab, c).reshape(*a.shape[:-2], 8, 8)

def bloch_from_rho(rho3, eps=1e-6):
    qubit_states = []
    bloch_vectors = []
    for i, j in PAIRS:
        rii = rho3[..., i, i].real.clamp(min=0)
        rjj = rho3[..., j, j].real.clamp(min=0)
        rij = rho3[..., i, j]
        tr = (rii + rjj).clamp(min=eps)
        n = torch.stack([2 * rij.real / tr, -2 * rij.imag / tr, (rii - rjj) / tr], dim=-1)
        n = n / n.norm(dim=-1, keepdim=True).clamp(min=1.0)
        nx, ny, nz = n[..., 0], n[..., 1], n[..., 2]
        q = 0.5 * torch.stack([
            torch.stack([torch.complex(1 + nz, torch.zeros_like(nz)), torch.complex(nx, -ny)], dim=-1),
            torch.stack([torch.complex(nx, ny), torch.complex(1 - nz, torch.zeros_like(nz))], dim=-1),
        ], dim=-2)
        qubit_states.append(q)
        bloch_vectors.append(n)
    return torch.stack(qubit_states, dim=2), torch.cat(bloch_vectors, dim=-1)

def cnot_3q(control, target):
    U = torch.zeros(8, 8, dtype=torch.complex64)
    for idx in range(8):
        bits = [(idx >> (2 - q)) & 1 for q in range(3)]
        if bits[control] == 1:
            bits[target] ^= 1
        out = (bits[0] << 2) | (bits[1] << 1) | bits[2]
        U[out, idx] = 1.0
    return U

class ThreeQubitUnitary(nn.Module):
    def __init__(self, n_layers=2, entangle=True):
        super().__init__()
        self.n_layers = n_layers
        self.entangle = entangle
        self.theta = nn.Parameter(0.05 * torch.randn(n_layers, 3, 3))
        self.register_buffer("c01", cnot_3q(0, 1), persistent=False)
        self.register_buffer("c12", cnot_3q(1, 2), persistent=False)
        self.register_buffer("c20", cnot_3q(2, 0), persistent=False)

    def matrix(self):
        U = torch.eye(8, dtype=torch.complex64, device=self.theta.device)
        for layer_idx in range(self.n_layers):
            qs = [rz(self.theta[layer_idx, q, 2]) @ ry(self.theta[layer_idx, q, 1]) @ rz(self.theta[layer_idx, q, 0]) for q in range(3)]
            layer = torch.kron(torch.kron(qs[0], qs[1]), qs[2])
            U = layer @ U
            if self.entangle:
                U = self.c20 @ self.c12 @ self.c01 @ U
        return U

class Arch2QuantumFeature(nn.Module):
    def __init__(self, n_layers=2, entangle=True):
        super().__init__()
        self.unitary = ThreeQubitUnitary(n_layers=n_layers, entangle=entangle)
        self.register_buffer("obs", ARCH2_OBS.clone(), persistent=False)

    def forward(self, rho3):
        qubits, _ = bloch_from_rho(rho3)
        rho8 = kron3(qubits[:, :, 0], qubits[:, :, 1], qubits[:, :, 2])
        U = self.unitary.matrix()
        evolved = U @ rho8 @ U.conj().transpose(-1, -2)
        return torch.einsum("btij,kji->btk", evolved, self.obs).real.clamp(-1.0, 1.0)

class Arch2ControlFeature(nn.Module):
    def __init__(self, hidden=4, out_dim=12):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(9, hidden), nn.Tanh(), nn.Linear(hidden, out_dim), nn.Tanh())

    def forward(self, rho3):
        _, bloch = bloch_from_rho(rho3)
        return self.net(bloch)

class SharedPoolHead(nn.Module):
    def __init__(self, in_dim, d_model=32, heads=4, dropout=0.10, num_tokens=36):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, num_tokens + 1, d_model))
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=2 * d_model,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=1)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
        nn.init.normal_(self.pos, std=0.02)
        nn.init.normal_(self.cls, std=0.02)

    def forward(self, features, rho=None):
        B = features.shape[0]
        x = self.embed(features)
        x = torch.cat([self.cls.expand(B, -1, -1), x], dim=1)
        x = x + self.pos[:, :x.shape[1]]
        return self.head(self.encoder(x)[:, 0]).squeeze(-1)

class FidelityPoolHead(nn.Module):
    # Single-layer attention pooler with density-fidelity bias S_ij = Re Tr(rho_i rho_j).
    def __init__(self, in_dim, d_model=32, heads=4, dropout=0.10, num_tokens=36, alpha_init=0.10, center=True):
        super().__init__()
        if d_model % heads != 0:
            raise ValueError("d_model must be divisible by heads")
        self.d_model = d_model
        self.heads = heads
        self.dk = d_model // heads
        self.center = center
        self.embed = nn.Linear(in_dim, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, num_tokens + 1, d_model))
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 2 * d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d_model, d_model))
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
        self.alpha = nn.Parameter(torch.tensor(float(alpha_init)))
        nn.init.normal_(self.pos, std=0.02)
        nn.init.normal_(self.cls, std=0.02)

    def _fidelity_bias(self, rho, n_total):
        # rho: [B,T,3,3], complex. Bias returned as [B,1,N,N]. CLS row/col remains zero.
        S = torch.einsum("btij,bsji->bts", rho, rho).real
        if self.center:
            S = S - S.mean(dim=(-1, -2), keepdim=True)
        B, T, _ = S.shape
        bias = torch.zeros(B, n_total, n_total, dtype=S.dtype, device=S.device)
        bias[:, 1:T+1, 1:T+1] = S
        return bias.unsqueeze(1)

    def forward(self, features, rho=None):
        if rho is None:
            raise ValueError("FidelityPoolHead requires rho for density-fidelity attention bias.")
        B = features.shape[0]
        x = self.embed(features)
        x = torch.cat([self.cls.expand(B, -1, -1), x], dim=1)
        x = x + self.pos[:, :x.shape[1]]
        residual = x
        x_norm = self.norm1(x)
        qkv = self.qkv(x_norm).reshape(B, x.shape[1], 3, self.heads, self.dk).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        logits = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.dk)
        logits = logits + self.alpha * self._fidelity_bias(rho, x.shape[1])
        attn = torch.softmax(logits, dim=-1)
        attended = torch.matmul(attn, v).transpose(1, 2).reshape(B, x.shape[1], self.d_model)
        x = residual + self.drop(self.out(attended))
        x = x + self.drop(self.ff(self.norm2(x)))
        return self.head(x[:, 0]).squeeze(-1)

class TinyClassicalViT(nn.Module):
    def __init__(self, token_dim=8, cfg: Exp37Config = None):
        super().__init__()
        cfg = cfg or CFG
        self.pool = SharedPoolHead(token_dim, cfg.d_model, cfg.heads, cfg.dropout, cfg.patch_grid ** 2)

    def forward(self, tokens):
        return self.pool(tokens)

class FeatureThenPool(nn.Module):
    def __init__(self, feature_extractor, feature_dim, cfg: Exp37Config):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.pool = SharedPoolHead(feature_dim, cfg.d_model, cfg.heads, cfg.dropout, cfg.patch_grid ** 2)

    def forward(self, rho):
        return self.pool(self.feature_extractor(rho))

class TokenArch2Fusion(nn.Module):
    def __init__(self, feature_extractor, cfg: Exp37Config, use_fidelity=False):
        super().__init__()
        self.feature_extractor = feature_extractor
        in_dim = cfg.token_dim + cfg.arch2_obs_dim
        if use_fidelity:
            self.pool = FidelityPoolHead(
                in_dim, cfg.d_model, cfg.heads, cfg.dropout, cfg.patch_grid ** 2,
                cfg.fidelity_init_alpha, cfg.fidelity_center,
            )
        else:
            self.pool = SharedPoolHead(in_dim, cfg.d_model, cfg.heads, cfg.dropout, cfg.patch_grid ** 2)

    def forward(self, rho, tokens):
        q_features = self.feature_extractor(rho)
        features = torch.cat([tokens, q_features], dim=-1)
        return self.pool(features, rho=rho)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Single registry. Every variant is instantiated in NOTEBOOK_SELF_TEST.
MODEL_REGISTRY: Dict[str, Tuple[Callable[[Exp37Config], nn.Module], str]] = {
    "TokenBaseline_TinyViT": (
        lambda c: TinyClassicalViT(c.token_dim, c), "tokens"
    ),
    "Arch2_Control": (
        lambda c: FeatureThenPool(Arch2ControlFeature(c.control_hidden_a2, c.arch2_obs_dim), c.arch2_obs_dim, c), "rho"
    ),
    "Arch2_Quantum_Ent": (
        lambda c: FeatureThenPool(Arch2QuantumFeature(c.q_layers, entangle=True), c.arch2_obs_dim, c), "rho"
    ),
    "Token_Arch2Control_Fusion": (
        lambda c: TokenArch2Fusion(Arch2ControlFeature(c.control_hidden_a2, c.arch2_obs_dim), c, use_fidelity=False), "both"
    ),
    "Token_Arch2Quantum_Fusion": (
        lambda c: TokenArch2Fusion(Arch2QuantumFeature(c.q_layers, entangle=True), c, use_fidelity=False), "both"
    ),
    "Token_Arch2Quantum_Fusion_FidelityAttention": (
        lambda c: TokenArch2Fusion(Arch2QuantumFeature(c.q_layers, entangle=True), c, use_fidelity=True), "both"
    ),
}

for name, (builder, input_key) in MODEL_REGISTRY.items():
    model = builder(CFG)
    print(f"{name:48s} input={input_key:6s} params={count_params(model)}")


In [ ]:
# =========================
# CELL 9 — Training, prediction, metrics
# =========================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan"),
        "avg_precision": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan"),
        "false_alarm_rejection": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
    }

def make_loader(split, input_key, cfg: Exp37Config, shuffle=True):
    rho, tokens, y = DATA[split]
    if input_key == "rho":
        ds = torch.utils.data.TensorDataset(torch.tensor(rho), torch.tensor(y))
    elif input_key == "tokens":
        ds = torch.utils.data.TensorDataset(torch.tensor(tokens), torch.tensor(y))
    elif input_key == "both":
        ds = torch.utils.data.TensorDataset(torch.tensor(rho), torch.tensor(tokens), torch.tensor(y))
    else:
        raise ValueError(f"Unknown input_key={input_key}")
    return torch.utils.data.DataLoader(ds, batch_size=cfg.batch_size, shuffle=shuffle)

def forward_batch(model, batch, input_key, device):
    if input_key == "rho":
        rho, y = batch
        return model(rho.to(device)), y.to(device)
    if input_key == "tokens":
        tokens, y = batch
        return model(tokens.to(device)), y.to(device)
    if input_key == "both":
        rho, tokens, y = batch
        return model(rho.to(device), tokens.to(device)), y.to(device)
    raise ValueError(f"Unknown input_key={input_key}")

def predict_model(model, split, input_key, device):
    rho, tokens, y = DATA[split]
    probs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(y), 512):
            if input_key == "rho":
                logits = model(torch.tensor(rho[i:i+512]).to(device))
            elif input_key == "tokens":
                logits = model(torch.tensor(tokens[i:i+512]).to(device))
            elif input_key == "both":
                logits = model(torch.tensor(rho[i:i+512]).to(device), torch.tensor(tokens[i:i+512]).to(device))
            else:
                raise ValueError(input_key)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(probs), y

def train_eval_model(builder, input_key, cfg: Exp37Config, seed, model_name="model"):
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = builder(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(max(float(cfg.pos_weight), 1e-6), device=device))
    loader = make_loader("train", input_key, cfg, shuffle=True)

    for epoch in range(cfg.epochs):
        model.train()
        for batch in loader:
            optimizer.zero_grad(set_to_none=True)
            logits, yb = forward_batch(model, batch, input_key, device)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

    probs, y_val = predict_model(model, "val", input_key, device)
    metrics = compute_metrics(y_val, probs)
    metrics.update({"seed": seed, "model": model_name})
    torch.save({"model_state": model.state_dict(), "cfg": asdict(cfg), "seed": seed, "model_name": model_name}, RUN_DIR / "ckpt" / f"{model_name}_seed{seed}.pt")
    return metrics


In [ ]:
# =========================
# CELL 10 — Run grid and save per-seed CSV
# =========================
rows = []
if CFG.run_experiment:
    for model_name, (builder, input_key) in MODEL_REGISTRY.items():
        model_rows = []
        for seed in CFG.seeds:
            metrics = train_eval_model(builder, input_key, CFG, seed, model_name=model_name)
            rows.append(metrics)
            model_rows.append(metrics)
        sub = pd.DataFrame(model_rows)
        print(
            f"{model_name:48s} "
            f"AUC={sub.roc_auc.mean():.4f}±{sub.roc_auc.std():.4f}  "
            f"AP={sub.avg_precision.mean():.4f}±{sub.avg_precision.std():.4f}  "
            f"F1={sub.f1.mean():.4f}±{sub.f1.std():.4f}"
        )
    results_df = pd.DataFrame(rows)
else:
    print("RUN_EXPERIMENT=False: skipping training grid.")
    results_df = pd.DataFrame(columns=["model", "seed", "roc_auc", "avg_precision", "f1"])

per_seed_csv = RUN_DIR / "logs" / "grid_per_seed_metrics.csv"
results_df.to_csv(per_seed_csv, index=False)
print("Saved:", per_seed_csv)


In [ ]:
# =========================
# CELL 11 — Paired comparisons and paper tables
# =========================
def paired_comparison(df, metric, first, second):
    pivot = df.pivot_table(index="seed", columns="model", values=metric)
    if first not in pivot.columns or second not in pivot.columns:
        return None
    aligned = pivot[[first, second]].dropna()
    if len(aligned) == 0:
        return None
    delta = (aligned[first] - aligned[second]).to_numpy()
    if np.all(delta == 0):
        wilcox_2s = wilcox_1s = 1.0
    else:
        wilcox_2s = wilcoxon(delta, alternative="two-sided").pvalue
        wilcox_1s = wilcoxon(delta, alternative="greater").pvalue
    ttest = ttest_1samp(delta, 0, alternative="greater").pvalue if len(delta) > 1 else float("nan")
    return {
        "metric": metric,
        "comparison": f"{first} - {second}",
        "first": first,
        "second": second,
        "n": len(delta),
        "mean_delta": float(delta.mean()),
        "std_delta": float(delta.std(ddof=1)) if len(delta) > 1 else 0.0,
        "positive_seeds": int((delta > 0).sum()),
        "wilcoxon_2s": float(wilcox_2s),
        "wilcoxon_1s": float(wilcox_1s),
        "ttest_1s": float(ttest),
    }

def sig_star(p):
    if not np.isfinite(p): return "n/a"
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return "n.s."

COMPARISONS = [
    ("Arch2_Quantum_Ent", "Arch2_Control"),
    ("Token_Arch2Quantum_Fusion", "Token_Arch2Control_Fusion"),
    ("Token_Arch2Quantum_Fusion", "TokenBaseline_TinyViT"),
    ("Token_Arch2Quantum_Fusion_FidelityAttention", "Token_Arch2Quantum_Fusion"),
    ("Token_Arch2Quantum_Fusion_FidelityAttention", "Token_Arch2Control_Fusion"),
    ("Token_Arch2Quantum_Fusion_FidelityAttention", "TokenBaseline_TinyViT"),
    ("Token_Arch2Quantum_Fusion_FidelityAttention", "Arch2_Quantum_Ent"),
]

if len(results_df) > 0:
    comparison_rows = []
    for metric in ["roc_auc", "avg_precision", "f1"]:
        for first, second in COMPARISONS:
            row = paired_comparison(results_df, metric, first, second)
            if row is not None:
                comparison_rows.append(row)
    comparison_df = pd.DataFrame(comparison_rows)
else:
    comparison_df = pd.DataFrame()

comparison_csv = RUN_DIR / "logs" / "paired_comparisons.csv"
comparison_df.to_csv(comparison_csv, index=False)
print("Saved:", comparison_csv)

# Table A
if len(results_df) > 0:
    param_counts = {name: count_params(builder(CFG)) for name, (builder, _) in MODEL_REGISTRY.items()}
    table_rows = []
    for name in MODEL_REGISTRY.keys():
        s = results_df[results_df.model == name]
        table_rows.append({
            "model": name,
            "params": param_counts[name],
            "AP": f"{s.avg_precision.mean():.4f}±{s.avg_precision.std():.4f}",
            "AUC": f"{s.roc_auc.mean():.4f}±{s.roc_auc.std():.4f}",
            "F1": f"{s.f1.mean():.4f}±{s.f1.std():.4f}",
            "precision": f"{s.precision.mean():.3f}",
            "recall": f"{s.recall.mean():.3f}",
            "FAR": f"{s.false_alarm_rejection.mean():.3f}",
        })
    tableA = pd.DataFrame(table_rows)
else:
    tableA = pd.DataFrame()

tableA_csv = RUN_DIR / "logs" / "tableA_verifier_comparison.csv"
tableA.to_csv(tableA_csv, index=False)
print("=== Table A: EXP37 verifier comparison ===")
display(tableA)
print("Saved:", tableA_csv)

# Table B: AUC comparisons
if len(comparison_df) > 0:
    auc_df = comparison_df[comparison_df.metric == "roc_auc"].copy()
    tableB = pd.DataFrame({
        "comparison": auc_df["comparison"],
        "metric": "AUC",
        "delta": auc_df.apply(lambda r: f"{r.mean_delta:+.4f}±{r.std_delta:.4f}", axis=1),
        "pos": auc_df.apply(lambda r: f"{int(r.positive_seeds)}/{int(r.n)}", axis=1),
        "wilcoxon_2s": auc_df["wilcoxon_2s"].map(lambda x: f"{x:.3f}"),
        "ttest_1s": auc_df["ttest_1s"].map(lambda x: f"{x:.3f}" if np.isfinite(x) else "n/a"),
        "sig": auc_df["ttest_1s"].map(sig_star),
    })
else:
    tableB = pd.DataFrame()

tableB_csv = RUN_DIR / "logs" / "tableB_auc_significance.csv"
tableB.to_csv(tableB_csv, index=False)
print("=== Table B: AUC paired deltas ===")
display(tableB)
print("Saved:", tableB_csv)


In [ ]:
# =========================
# CELL 12 — NOTEBOOK_SELF_TEST
# =========================
def NOTEBOOK_SELF_TEST():
    print("Running NOTEBOOK_SELF_TEST...")
    report = []

    def ok(name):
        print("PASS:", name)
        report.append({"check": name, "status": "PASS"})

    def fail(name, message):
        report.append({"check": name, "status": "FAIL", "message": str(message)})
        raise AssertionError(f"{name}: {message}")

    # 1. Single config class and no legacy experiment config dependency.
    if not isinstance(CFG, Exp37Config):
        fail("single Exp37Config instance", type(CFG))
    legacy_config_symbols = [name for name in globals() if name.endswith("Config") and name != "Exp37Config"]
    if legacy_config_symbols:
        fail("no legacy experiment config symbols", legacy_config_symbols)
    ok("single Exp37Config instance; no legacy experiment config symbols")

    # 2. Output/log paths.
    for p in [RUN_DIR, RUN_DIR / "logs", RUN_DIR / "ckpt"]:
        if not p.exists():
            fail("output path creation", f"Missing {p}")
    test_csv = RUN_DIR / "logs" / "self_test_write.csv"
    pd.DataFrame([{"ok": 1}]).to_csv(test_csv, index=False)
    if not test_csv.exists():
        fail("CSV/log writing", "self_test_write.csv missing")
    ok("output path creation and CSV/log writing")

    # 3. Dataset interface.
    for split in ["train", "val"]:
        if split not in DATA:
            fail("DATA split exists", split)
        rho, tokens, y = DATA[split]
        if rho.ndim != 4 or rho.shape[-2:] != (3, 3):
            fail("rho shape", rho.shape)
        if tokens.ndim != 3 or tokens.shape[-1] != CFG.token_dim:
            fail("token shape", tokens.shape)
        if len(y) != rho.shape[0] or len(y) != tokens.shape[0]:
            fail("dataset length alignment", (rho.shape, tokens.shape, y.shape))
    ok("dataset interface: rho/tokens/y shapes aligned")

    # 4. Dummy forward pass for every registered model.
    device = "cuda" if torch.cuda.is_available() else "cpu"
    B = 3
    T = CFG.patch_grid ** 2
    rho = torch.tensor(DATA["val"][0][:B]).to(device)
    tokens = torch.tensor(DATA["val"][1][:B]).to(device)
    if rho.shape[0] < B:
        rho = torch.eye(3, dtype=torch.complex64, device=device).reshape(1, 1, 3, 3).repeat(B, T, 1, 1) / 3.0
        tokens = torch.randn(B, T, CFG.token_dim, device=device)

    for name, (builder, input_key) in MODEL_REGISTRY.items():
        model = builder(CFG).to(device)
        model.eval()
        with torch.no_grad():
            if input_key == "rho":
                out = model(rho)
            elif input_key == "tokens":
                out = model(tokens)
            elif input_key == "both":
                out = model(rho, tokens)
            else:
                fail("valid input_key", f"{name}: {input_key}")
        if out.shape != (B,):
            fail("dummy forward output shape", f"{name}: got {tuple(out.shape)}, expected {(B,)}")
        if not torch.isfinite(out).all():
            fail("dummy forward finite logits", name)
    ok("dummy forward pass for every model variant")

    # 5. Metrics pipeline.
    y_true = np.array([0, 1, 0, 1])
    y_prob = np.array([0.1, 0.9, 0.2, 0.8])
    metrics = compute_metrics(y_true, y_prob)
    for key in ["roc_auc", "avg_precision", "f1", "precision", "recall"]:
        if key not in metrics:
            fail("metrics key exists", key)
    ok("metrics pipeline")

    # 6. Result CSVs exist after run-all.
    required_csvs = [
        RUN_DIR / "logs" / "grid_per_seed_metrics.csv",
        RUN_DIR / "logs" / "paired_comparisons.csv",
        RUN_DIR / "logs" / "tableA_verifier_comparison.csv",
        RUN_DIR / "logs" / "tableB_auc_significance.csv",
    ]
    for p in required_csvs:
        if not p.exists():
            fail("result CSV exists", p)
    ok("result CSV files exist")

    # 7. Honest not-tested marker.
    if CFG.dry_run:
        print("NOT TESTED: real Google Drive/FPSD data loading, YOLO inference, raster TIF extraction, full 10-seed training.")
    else:
        print("FULL MODE: dry-run bypass disabled. Make sure Drive/FPSD paths above were resolved correctly.")

    report_df = pd.DataFrame(report)
    report_path = RUN_DIR / "logs" / "self_test_report.csv"
    report_df.to_csv(report_path, index=False)
    print("SELF-TEST PASSED ✅")
    print("Self-test report:", report_path)
    return report_df

self_test_report = NOTEBOOK_SELF_TEST()
display(self_test_report)


## Validation policy

This notebook was built so that a fresh `Runtime → Run all` passes in `DRY_RUN=True` mode. The dry-run validates code order, model registry, dummy and real synthetic forward passes, metrics, CSV writing, and output-folder creation.

For real FPSD execution, set `DRY_RUN=False`. Real YOLO inference and Drive/FPSD raster extraction are intentionally not claimed as locally tested in this environment.


In [ ]:
# Inspect artifacts created by this notebook, not a fixed historical run folder.
from pathlib import Path
logfiles = sorted((RUN_DIR / "logs").glob("*"))
print("Run log files:", len(logfiles))
for file in logfiles:
    print(file.relative_to(RUN_DIR))


In [ ]:
# Display the resolved configuration for this run.
from pprint import pprint
with (RUN_DIR / "logs" / "config_resolved.json").open("r", encoding="utf-8") as handle:
    pprint(json.load(handle))
